# 00 - EDA Lite (Memory Efficient)
## Business Entity Resolution Challenge

Memory-efficient version that samples data instead of loading everything

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Imports successful")

✓ Imports successful


## 1. Load Data (First 1000 rows for quick EDA)

In [2]:
data_root = Path('../../../student_resource/dataset')
train_dir = data_root / 'train'
test_dir = data_root / 'test'

# Load samples
train_s1 = pd.read_csv(train_dir / 'train_source1.tsv', sep='\t', nrows=1000)
train_s2 = pd.read_csv(train_dir / 'train_source2.tsv', sep='\t', nrows=1000)
train_s3 = pd.read_csv(train_dir / 'train_source3.tsv', sep='\t', nrows=1000)
train_gt = pd.read_csv(train_dir / 'train_ground_truth.tsv', sep='\t', nrows=1000)

print("Sample data loaded (1000 rows each)")
print(f"\nColumns in Source 1: {train_s1.columns.tolist()}")
print(f"\nFirst record:")
print(train_s1.head(1).T)

Sample data loaded (1000 rows each)

Columns in Source 1: ['entity_id', 'business_name', 'business_address', 'country']

First record:
                                                       0
entity_id                                   S1-925783039
business_name                        Orelee's Barbershop
business_address  1795 Westchester Drive, High Point, NC
country                                               US


## 2. Get Full Dataset Statistics (Efficient)

In [3]:
# Count rows efficiently without loading all data
def count_lines(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f) - 1  # Subtract header

train_s1_count = count_lines(train_dir / 'train_source1.tsv')
train_s2_count = count_lines(train_dir / 'train_source2.tsv')
train_s3_count = count_lines(train_dir / 'train_source3.tsv')

print("Full Training Dataset Sizes:")
print(f"  Source 1: {train_s1_count:,} entities")
print(f"  Source 2: {train_s2_count:,} entities")
print(f"  Source 3: {train_s3_count:,} entities")
print(f"  Total: {train_s1_count + train_s2_count + train_s3_count:,} entities")

Full Training Dataset Sizes:
  Source 1: 2,206,821 entities
  Source 2: 5,034,616 entities
  Source 3: 5,285,603 entities
  Total: 12,527,040 entities


## 3. Country Distribution (Sample)

In [4]:
print("Country Distribution (from sample):")
print("\nSource 1:")
print(train_s1['country'].value_counts())
print("\nSource 2:")
print(train_s2['country'].value_counts())
print("\nSource 3:")
print(train_s3['country'].value_counts())

Country Distribution (from sample):

Source 1:
country
US       595
India    405
Name: count, dtype: int64

Source 2:
country
US       604
India    396
Name: count, dtype: int64

Source 3:
country
US       593
India    407
Name: count, dtype: int64


## 4. Ground Truth Statistics (Sample)

In [5]:
def parse_matches(match_str):
    if pd.isna(match_str) or str(match_str).strip() == '':
        return []
    return [s.strip() for s in str(match_str).split(',') if s.strip()]

train_gt['matches'] = train_gt['matched_entity_ids'].apply(parse_matches)
train_gt['num_matches'] = train_gt['matches'].apply(len)

print("Ground Truth Statistics (from sample):")
print(f"\nMatch count distribution:")
print(train_gt['num_matches'].value_counts().sort_index())

singleton_pct = 100 * (train_gt['num_matches'] == 0).sum() / len(train_gt)
print(f"\nSingletons: {(train_gt['num_matches'] == 0).sum()} ({singleton_pct:.1f}%)")
print(f"Has matches: {(train_gt['num_matches'] > 0).sum()} ({100-singleton_pct:.1f}%)")

Ground Truth Statistics (from sample):

Match count distribution:
num_matches
0     47
1     55
2    178
3    248
4    222
5    143
6     68
7     35
8      3
9      1
Name: count, dtype: int64

Singletons: 47 (4.7%)
Has matches: 953 (95.3%)


## 5. Sample True Positive Pairs

In [6]:
has_matches = train_gt[train_gt['num_matches'] > 0].head(5)

print("Sample True Positive Pairs:")
print("=" * 80)

for idx, row in has_matches.iterrows():
    s1_id = row['source1_entity_id']
    s1_rec = train_s1[train_s1['entity_id'] == s1_id]
    
    if len(s1_rec) > 0:
        s1_rec = s1_rec.iloc[0]
        print(f"\n[{s1_id}] {s1_rec['business_name']}")
        print(f"  Address: {s1_rec['business_address']}")
        print(f"  Country: {s1_rec['country']}")
        print(f"  Matches: {', '.join(row['matches'][:3])}")
        print("-" * 80)

Sample True Positive Pairs:


## 6. Noise Pattern Examples

In [7]:
print("Sample Business Names (showing variety and noise):")
print("=" * 80)
for i, row in train_s1.head(10).iterrows():
    print(f"{row['entity_id']:15s} | {row['business_name'][:60]}")

print("\n" + "=" * 80)
print("Common Noise Patterns to Handle:")
print("  - Legal suffixes: Inc, Corp, LLC, Pvt, Ltd, SARL, SAS")
print("  - Punctuation: & vs 'and', periods, commas")
print("  - Abbreviations: St/Street, Rd/Road, Ave/Avenue")
print("  - Address variations: missing ZIP, state, landmark references")

Sample Business Names (showing variety and noise):
S1-925783039    | Orelee's Barbershop
S1-773889195    | Prime Money
S1-377745466    | B+ Retail Inc
S1-133037285    | Christ Chapel
S1-755362802    | Prabhav Business Center
S1-851869949    | Custom Wealth Services LLC
S1-785847572    | Consulting Nyasa Nursing Private Limited
S1-27541239     | Nexus Anchor Rain
S1-629417405    | Moore Bitwise Inc
S1-22305073     | Dermatology Green Medicine

Common Noise Patterns to Handle:
  - Legal suffixes: Inc, Corp, LLC, Pvt, Ltd, SARL, SAS
  - Punctuation: & vs 'and', periods, commas
  - Abbreviations: St/Street, Rd/Road, Ave/Avenue
  - Address variations: missing ZIP, state, landmark references


## 7. Summary

In [8]:
print("\n" + "=" * 80)
print("EDA SUMMARY")
print("=" * 80)
print("\n✓ Dataset is large (~12.5M total records)")
print("✓ Data structure: entity_id, business_name, business_address, country")
print("✓ Multiple matches per S1 entity (0 to many)")
print("✓ Singletons present (entities with no matches)")
print("✓ Noisy data with variations in names and addresses")
print("\nReady to proceed with training pipeline!")
print("=" * 80)


EDA SUMMARY

✓ Dataset is large (~12.5M total records)
✓ Data structure: entity_id, business_name, business_address, country
✓ Multiple matches per S1 entity (0 to many)
✓ Singletons present (entities with no matches)
✓ Noisy data with variations in names and addresses

Ready to proceed with training pipeline!
